<a href="https://colab.research.google.com/github/lmbernardo7520112/desafio_bairesdev_embedded_vision_project_LMB/blob/main/FER2013_MobileNetV2_EfficientNet_Pruning_%2B_Quantiza%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
! ls /content/drive/MyDrive/Colab\ Notebooks/kaggle.json

'/content/drive/MyDrive/Colab Notebooks/kaggle.json'


In [8]:
# Instalar Kaggle API se não tiver
!pip install -q kaggle

# Autenticar (faça upload do kaggle.json no Colab antes)
# /content/drive/MyDrive/Colab\ Notebooks/kaggle.json
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Colab\ Notebooks/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Baixar dataset FER2013
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d data/


Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
  0% 0.00/60.3M [00:00<?, ?B/s]
100% 60.3M/60.3M [00:00<00:00, 646MB/s]


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, BatchNormalization, ReLU
from tensorflow.keras.layers import MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout

num_labels = 7   # emoções
width, height = 48, 48

def MobileNet_Light(input_shape=(width, height, 1), num_classes=num_labels):
    model = Sequential(name="MobileNet_Light")

    # Bloco inicial
    model.add(Conv2D(32, (3,3), strides=(2,2), padding='same',
                     input_shape=input_shape, use_bias=False))
    model.add(BatchNormalization())
    model.add(ReLU(6.))  # ReLU6 para compatibilidade com quantização

    # Blocos Depthwise Separable
    def depthwise_block(filters, stride):
        model.add(DepthwiseConv2D((3,3), strides=(stride,stride), padding='same', use_bias=False))
        model.add(BatchNormalization())
        model.add(ReLU(6.))
        model.add(Conv2D(filters, (1,1), strides=(1,1), padding='same', use_bias=False))
        model.add(BatchNormalization())
        model.add(ReLU(6.))

    depthwise_block(64, 1)
    depthwise_block(128, 2)
    depthwise_block(128, 1)
    depthwise_block(256, 2)
    depthwise_block(256, 1)
    depthwise_block(512, 2)

    # Pooling + Classificação
    model.add(GlobalAveragePooling2D())
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))

    return model

# Criar e compilar
model = MobileNet_Light()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "MobileNet_Light"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 24, 24, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 24, 24, 32)     │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 12, 12, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12, 12, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 12, 12, 128)    │         1,152 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 276,615 (1.06 MB)

 Trainable params: 272,135 (1.04 MB)

 Non-trainable params: 4,480 (17.50 KB)

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = 48  # FER2013 é 48x48
batch_size = 64

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2  # separa parte do treino como validação
)

train_generator = train_datagen.flow_from_directory(
    "data/train",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    class_mode="categorical",
    batch_size=batch_size,
    subset="training"
)

val_generator = train_datagen.flow_from_directory(
    "data/train",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    class_mode="categorical",
    batch_size=batch_size,
    subset="validation"
)


Found 22968 images belonging to 7 classes.
Found 5741 images belonging to 7 classes.


In [11]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(img_size, img_size, 3),  # precisa ser RGB
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # Congela base (vai liberar depois p/ fine-tuning)

model = models.Sequential([
    layers.Input(shape=(img_size, img_size, 1)),
    layers.Conv2D(3, (3,3), padding="same"),  # converte grayscale para 3 canais
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(train_generator.num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/tmp/ipython-input-380469413.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 48, 48, 3)      │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,981 (8.65 MB)

 Trainable params: 8,997 (35.14 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ModelCheckpoint("mobilenetv2_fer2013.h5", monitor="val_loss", save_best_only=True, verbose=1)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step - accuracy: 0.2048 - loss: 2.4017
Epoch 1: val_loss improved from inf to 1.80047, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 194s 521ms/step - accuracy: 0.2048 - loss: 2.4010 - val_accuracy: 0.2820 - val_loss: 1.8005 - learning_rate: 0.0010
Epoch 2/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - accuracy: 0.2623 - loss: 1.8725
Epoch 2: val_loss improved from 1.80047 to 1.74232, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 183s 508ms/step - accuracy: 0.2623 - loss: 1.8724 - val_accuracy: 0.2965 - val_loss: 1.7423 - learning_rate: 0.0010
Epoch 3/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.2725 - loss: 1.7914
Epoch 3: val_loss did not improve from 1.74232
359/359 ━━━━━━━━━━━━━━━━━━━━ 187s 522ms/step - accuracy: 0.2725 - loss: 1.7914 - val_accuracy: 0.2942 - val_loss: 1.7469 - learning_rate: 0.0010
Epoch 4/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step - accuracy: 0.2880 - loss: 1.7657
Epoch 4: val_loss improved from 1.74232 to 1.73684, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 180s 502ms/step - accuracy: 0.2880 - loss: 1.7657 - val_accuracy: 0.2954 - val_loss: 1.7368 - learning_rate: 0.0010
Epoch 5/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.2943 - loss: 1.7541
Epoch 5: val_loss improved from 1.73684 to 1.73515, saving model to mobilenetv2_fer2013.h5


...

359/359 ━━━━━━━━━━━━━━━━━━━━ 185s 516ms/step - accuracy: 0.3136 - loss: 1.7072 - val_accuracy: 0.3212 - val_loss: 1.6962 - learning_rate: 5.0000e-04
Epoch 20/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.3199 - loss: 1.6986
Epoch 20: val_loss did not improve from 1.69621
359/359 ━━━━━━━━━━━━━━━━━━━━ 179s 499ms/step - accuracy: 0.3198 - loss: 1.6986 - val_accuracy: 0.3155 - val_loss: 1.6985 - learning_rate: 5.0000e-04
Epoch 21/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.3280 - loss: 1.6950
Epoch 21: val_loss did not improve from 1.69621
359/359 ━━━━━━━━━━━━━━━━━━━━ 182s 508ms/step - accuracy: 0.3280 - loss: 1.6950 - val_accuracy: 0.3161 - val_loss: 1.7007 - learning_rate: 5.0000e-04
Epoch 22/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 460ms/step - accuracy: 0.3174 - loss: 1.6951
Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 22: val_loss did not improve from 1.69621
359/359 ━━━━━━━━━━━━━━━━━━━━ 181s 503ms/step - accuracy: 0.3174 

359/359 ━━━━━━━━━━━━━━━━━━━━ 182s 508ms/step - accuracy: 0.3244 - loss: 1.7017 - val_accuracy: 0.3195 - val_loss: 1.6931 - learning_rate: 2.5000e-04
Epoch 24/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - accuracy: 0.3243 - loss: 1.6891
Epoch 24: val_loss improved from 1.69310 to 1.69123, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 182s 506ms/step - accuracy: 0.3243 - loss: 1.6891 - val_accuracy: 0.3249 - val_loss: 1.6912 - learning_rate: 2.5000e-04
Epoch 25/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 467ms/step - accuracy: 0.3316 - loss: 1.6808
Epoch 25: val_loss improved from 1.69123 to 1.68564, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 184s 512ms/step - accuracy: 0.3316 - loss: 1.6808 - val_accuracy: 0.3188 - val_loss: 1.6856 - learning_rate: 2.5000e-04
Epoch 26/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - accuracy: 0.3279 - loss: 1.6890
Epoch 26: val_loss did not improve from 1.68564
359/359 ━━━━━━━━━━━━━━━━━━━━ 182s 507ms/step - accuracy: 0.3279 - loss: 1.6890 - val_accuracy: 0.3256 - val_loss: 1.6915 - learning_rate: 2.5000e-04
Epoch 27/50
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - accuracy: 0.3217 - loss: 1.6920
Epoch 27: val_loss improved from 1.68564 to 1.67813, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 182s 506ms/step - accuracy: 0.3217 - loss: 1.6920 - val_accuracy: 0.3285 - val_loss: 1.6781 - learning_rate: 2.5000e-04
Epoch 28/50
212/359 ━━━━━━━━━━━━━━━━━━━━ 1:06 453ms/step - accuracy: 0.3359 - loss: 1.6750